# Notebook 02: Exploratory data analysis and feature analysis

Derived from **`mmr comprehensive.ipynb`**. This notebook **explores** the preprocessed panel saved by Notebook 01: global trends, bivariate patterns, distributions, missingness (if any), and **correlation** structure among features and MMR.

**Prerequisite:** Run Notebook 01 so `country_year_modeling_panel_cleaned.csv` exists under `../data/processed/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

df = pd.read_csv("../data/processed/country_year_modeling_panel_cleaned.csv")
df["year"] = df["year"].astype(int)
target = "log_mmr"
features = [
    "gdp_pc", "health_exp_gdp", "fertility", "skilled_births",
    "pm25", "heat_index35_days", "cooling_degree_days",
]
features = [f for f in features if f in df.columns]
print("Using features:", features)
df.head()


## Shape and dtypes


In [ ]:
print("Shape:", df.shape)
print(df.info())
#5712 instances and 12 variables
#no missing values
#all numeric predictors
#ISO3 and country are categorical identifiers


## Global trend and bivariate views (from comprehensive notebook)


In [ ]:
#global time trend of mmr
global_trend = df.groupby("year")["mmr"].mean().reset_index()

plt.figure(figsize=(8,5))
plt.plot(global_trend["year"], global_trend["mmr"])
plt.title("Global Average MMR Over Time")
plt.xlabel("Year")
plt.ylabel("Average MMR")
plt.show()


In [ ]:
#fertility vs mmr scatterplot
plt.figure(figsize=(7,5))
plt.scatter(df["fertility"], df["mmr"], c=df["gdp_pc"], cmap="viridis", alpha=0.6)
plt.colorbar(label="GDP per capita")
plt.xlabel("Fertility Rate")
plt.ylabel("MMR")
plt.title("Fertility vs MMR (Color = GDP)")
plt.show()


In [ ]:
#boxplot for income group comparison
df["income_group"] = pd.qcut(df["gdp_pc"], 4, labels=["Low", "Lower-Mid", "Upper-Mid", "High"])

plt.figure(figsize=(8,5))
sns.boxplot(x="income_group", y="mmr", data=df)
plt.title("MMR by Income Level")
plt.show()


In [ ]:
#outlier analysis
df.sort_values("mmr", ascending=False).head(10)


In [ ]:
#heat index vs mmr
plt.scatter(df["heat_index35_days"], df["mmr"], alpha=0.5)
plt.xlabel("Heat Index Days >35C(95F)")
plt.ylabel("MMR")
plt.title("Heat Stress vs MMR")
plt.show()


## Distributions and descriptive statistics for modeling columns


In [ ]:
# Distribution of MMR (raw)
plt.figure(figsize=(7,4))
plt.hist(df["mmr"].dropna(), bins=40)
plt.title("Distribution of MMR (raw)")
plt.xlabel("MMR")
plt.ylabel("Count")
plt.show()

# Distribution of log(MMR)
plt.figure(figsize=(7,4))
plt.hist(df[target].dropna(), bins=40)
plt.title("Distribution of log1p(MMR)")
plt.xlabel("log1p(MMR)")
plt.ylabel("Count")
plt.show()

# Basic summary stats
display(df[features + ["mmr", target]].describe())


## Missingness and correlation structure


In [ ]:
missing_counts = df[features].isna().sum().sort_values(ascending=False)
print("Missing counts:\n", missing_counts)

print("\nRows with any missing feature:", df[features].isna().any(axis=1).sum())


In [ ]:
#pandas
corr_matrix = df[features + ["mmr"]].corr(method="pearson")
corr_matrix


In [ ]:
tmp = df[features + ["mmr"]].copy()

# quick fill for correlation only (median)
tmp = tmp.apply(lambda s: s.fillna(s.median()) if s.dtype != "object" else s)

corr_pd = tmp.corr(method="pearson")
display(corr_pd)

corr_np = np.corrcoef(tmp.values.T)
corr_np_df = pd.DataFrame(corr_np, index=tmp.columns, columns=tmp.columns)
display(corr_np_df)

# Heatmap-style plot without seaborn (since you're okay with plain matplotlib)
plt.figure(figsize=(9,7))
plt.imshow(corr_pd.values, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_pd.columns)), corr_pd.columns, rotation=90)
plt.yticks(range(len(corr_pd.index)), corr_pd.index)
plt.title("Correlation Matrix (Pearson)")
plt.show()


### Interpretation notes (from original analysis)


In [ ]:
#Fertility has very strong positive correlation with MMR (0.82)
#GDP per capita has moderate negative correlation (-0.36)
#Skilled births negatively correlated (-0.33)
#PM2.5 moderately positively correlated (0.43)
#Climate variables show weaker correlations
#Fertility is the strongest linear predictor.


In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()
